# Компаративна анализа: Професор vs AI модели

Овој извештај ги споредува оценките на студентски проекти дадени од:
- **Професор** (референтна вредност)
- **Claude Opus** (најголем AI модел)
- **Claude Sonnet** (среден AI модел)
- **Claude Haiku** (најмал AI модел)

**Напомена:** Колоната `tocnost` (точност) е исклучена од анализата бидејќи за AI моделите е статична вредност (1.0).

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Поставки за визуелизација
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# Бои за секој евалуатор
COLORS = {'Професор': '#2ecc71', 'Opus': '#3498db', 'Sonnet': '#9b59b6', 'Haiku': '#e74c3c'}

## 1. Вчитување и подготовка на податоци

Податоците се вчитуваат од `evaluations.csv` и се чистат празните редови. Имињата на евалуаторите се преведуваат на македонски.

In [ ]:
# Вчитај податоци
df = pd.read_csv('../evaluations.csv')
df = df.dropna(subset=['project_id', 'evaluator'])

# Преведи имиња
evaluator_map = {'ПРОФЕСОР': 'Професор', 'AI-OPUS': 'Opus', 'AI-SONNET': 'Sonnet', 'AI-HAIKU': 'Haiku'}
df['evaluator'] = df['evaluator'].map(evaluator_map)

# Конвертирај нумерички колони
for col in ['metap', 'tocnost', 'kviz', 'vkupno']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Вкупно евалуации: {len(df)}")
print(f"Број на проекти: {df['project_id'].nunique()}")
print(f"Типови проекти: {df['project_type'].unique().tolist()}")

In [ ]:
# Креирај pivot табели за споредба
kviz_pivot = df.pivot_table(index='project_id', columns='evaluator', values='kviz').reset_index()
metap_pivot = df.pivot_table(index='project_id', columns='evaluator', values='metap').reset_index()
vkupno_pivot = df.pivot_table(index='project_id', columns='evaluator', values='vkupno').reset_index()

# Додај тип на проект
project_types = df[['project_id', 'project_type']].drop_duplicates().set_index('project_id')
for pivot in [kviz_pivot, metap_pivot, vkupno_pivot]:
    pivot['project_type'] = pivot['project_id'].map(project_types['project_type'])

---

## 2. Основна статистика

Оваа табела ги прикажува основните статистички мерки за секој евалуатор:
- **Број** - колку евалуации се направени
- **Просек** - средна вредност на оценките
- **Ст. дев.** - стандардна девијација (распределба на оценките)
- **Мин/Макс** - најниска и највисока оценка
- **Медијана** - средишна вредност (50% од оценките се под, 50% над)

In [ ]:
# Статистика за вкупно (финална оценка)
summary = df.groupby('evaluator')['vkupno'].agg(['count', 'mean', 'std', 'min', 'max', 'median']).round(2)
summary.columns = ['Број', 'Просек', 'Ст. дев.', 'Мин', 'Макс', 'Медијана']
summary = summary.reindex(['Професор', 'Opus', 'Sonnet', 'Haiku'])
print("Финална оценка (vkupno):")
summary

In [ ]:
# Статистика за квиз (пред корекции)
summary_kviz = df.groupby('evaluator')['kviz'].agg(['mean', 'std', 'min', 'max', 'median']).round(2)
summary_kviz.columns = ['Просек', 'Ст. дев.', 'Мин', 'Макс', 'Медијана']
summary_kviz = summary_kviz.reindex(['Професор', 'Opus', 'Sonnet', 'Haiku'])
print("Квиз оценка (kviz) - пред примена на фактори:")
summary_kviz

---

## 3. Споредба на метаподатоци (metap)

Колоната `metap` претставува фактор за метаподатоци (0-1) кој ја корегира оценката врз основа на:
- Број на наведени извори
- Квалитет на референцирање

Овој график ја покажува распределбата на `metap` факторот за секој евалуатор.

In [ ]:
# Статистика за metap
metap_stats = df.groupby('evaluator')['metap'].agg(['mean', 'std', 'min', 'max']).round(3)
metap_stats.columns = ['Просек', 'Ст. дев.', 'Мин', 'Макс']
metap_stats = metap_stats.reindex(['Професор', 'Opus', 'Sonnet', 'Haiku'])
print("Фактор за метаподатоци (metap):")
metap_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot за metap
order = ['Професор', 'Opus', 'Sonnet', 'Haiku']
colors = [COLORS[e] for e in order]
sns.boxplot(data=df, x='evaluator', y='metap', order=order, palette=colors, ax=axes[0])
axes[0].set_title('Распределба на metap фактор', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Евалуатор')
axes[0].set_ylabel('metap (0-1)')
axes[0].set_ylim(0.6, 1.05)

# Bar chart за просечен metap
means = df.groupby('evaluator')['metap'].mean().reindex(order)
bars = axes[1].bar(order, means.values, color=colors)
axes[1].set_title('Просечен metap фактор', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Евалуатор')
axes[1].set_ylabel('Просек')
axes[1].set_ylim(0.8, 1.0)
for bar, val in zip(bars, means.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
                f'{val:.3f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig('metap_споредба.png', dpi=150, bbox_inches='tight')
plt.show()

### Споредба на metap по проект

Оваа табела покажува како секој евалуатор го оценил `metap` факторот за секој проект. Разликите укажуваат на различни интерпретации на критериумите за метаподатоци.

In [ ]:
# Детална споредба на metap по проект
metap_comparison = metap_pivot[['project_id', 'Професор', 'Opus', 'Sonnet', 'Haiku']].copy()

# Пресметај разлики од професор
for model in ['Opus', 'Sonnet', 'Haiku']:
    metap_comparison[f'{model}_разлика'] = metap_comparison[model] - metap_comparison['Професор']

# Прикажи проекти каде има разлика
has_diff = metap_comparison[
    (metap_comparison['Opus_разлика'].abs() > 0.01) | 
    (metap_comparison['Sonnet_разлика'].abs() > 0.01) | 
    (metap_comparison['Haiku_разлика'].abs() > 0.01)
][['project_id', 'Професор', 'Opus', 'Sonnet', 'Haiku']]

print(f"Проекти со разлика во metap оценка ({len(has_diff)} од {len(metap_comparison)}):")
has_diff

---

## 4. Распределба на финални оценки

Овие графици ја покажуваат распределбата на финалните оценки (`vkupno`) за секој евалуатор.

- **Boxplot (лево)**: Покажува медијана (линија во средина), интерквартилен опсег (кутија), и екстремни вредности (точки)
- **Violin plot (десно)**: Покажува густина на оценките - поширок дел значи повеќе оценки во тој опсег

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = ['Професор', 'Opus', 'Sonnet', 'Haiku']
colors = [COLORS[e] for e in order]

# Boxplot
sns.boxplot(data=df, x='evaluator', y='vkupno', order=order, palette=colors, ax=axes[0])
axes[0].set_title('Распределба на финални оценки (boxplot)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Евалуатор')
axes[0].set_ylabel('Финална оценка (vkupno)')

# Violin plot
sns.violinplot(data=df, x='evaluator', y='vkupno', order=order, palette=colors, ax=axes[1], inner='box')
axes[1].set_title('Густина на оценки (violin plot)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Евалуатор')
axes[1].set_ylabel('Финална оценка (vkupno)')

plt.tight_layout()
plt.savefig('распределба_оценки.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 5. Корелација со оценки од Професор

**Корелација (Pearson r)** мери колку две варијабли се движат заедно:
- **r = 1.0**: Совршена позитивна корелација (идентични рангирања)
- **r = 0.0**: Нема корелација (случајна поврзаност)
- **r = -1.0**: Совршена негативна корелација (спротивни рангирања)

**p-вредност** покажува статистичка значајност (p < 0.05 = значајно).

In [ ]:
# Пресметај корелации
correlations = {}
for model in ['Opus', 'Sonnet', 'Haiku']:
    valid = vkupno_pivot[['Професор', model]].dropna()
    r, p = stats.pearsonr(valid['Професор'], valid[model])
    correlations[model] = {'Pearson r': round(r, 4), 'p-вредност': f'{p:.2e}', 'n': len(valid)}

corr_df = pd.DataFrame(correlations).T
corr_df.index.name = 'AI Модел'
print("Корелација на финални оценки со Професор:")
corr_df

### Scatter plots: Професор vs AI модели

Секој точка претставува еден проект. 
- **X-оска**: Оценка од Професор
- **Y-оска**: Оценка од AI модел
- **Црна испрекината линија**: Линија на совршено согласување (ако AI = Професор)
- **Обоена линија**: Регресиска линија (вистински тренд)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = ['Opus', 'Sonnet', 'Haiku']

for i, model in enumerate(models):
    valid = vkupno_pivot[['Професор', model, 'project_type']].dropna()
    
    # Scatter по тип на проект
    markers = {'noAI': 'o', 'onlyAI': 's', 'hybrid': '^'}
    for ptype, marker in markers.items():
        subset = valid[valid['project_type'] == ptype]
        axes[i].scatter(subset['Професор'], subset[model], 
                       alpha=0.7, s=60, marker=marker, label=ptype)
    
    # Линија на совршено согласување
    axes[i].plot([0, 100], [0, 100], 'k--', alpha=0.5, label='Совршено согласување')
    
    # Регресиска линија
    z = np.polyfit(valid['Професор'], valid[model], 1)
    p = np.poly1d(z)
    x_line = np.linspace(0, 100, 100)
    axes[i].plot(x_line, p(x_line), color=COLORS[model], linewidth=2, label='Тренд')
    
    r = stats.pearsonr(valid['Професор'], valid[model])[0]
    axes[i].set_title(f'Професор vs {model}\n(r = {r:.3f})', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Оценка од Професор')
    axes[i].set_ylabel(f'Оценка од {model}')
    axes[i].set_xlim(0, 100)
    axes[i].set_ylim(0, 100)
    axes[i].legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('корелација_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 6. Разлики во оценување (AI - Професор)

Овие графици ги покажуваат разликите помеѓу оценките од AI моделите и Професорот.
- **Позитивна разлика**: AI дал повисока оценка од Професор
- **Негативна разлика**: AI дал пониска оценка од Професор
- **Нула**: AI и Професор се согласуваат

**MAE (Mean Absolute Error)** = просечна апсолутна разлика (колку во просек се разликуваат оценките)

In [ ]:
# Пресметај разлики
for model in ['Opus', 'Sonnet', 'Haiku']:
    vkupno_pivot[f'{model}_разлика'] = vkupno_pivot[model] - vkupno_pivot['Професор']

# Статистика за разлики
diff_stats = pd.DataFrame({
    'AI Модел': ['Opus', 'Sonnet', 'Haiku'],
    'Просечна разлика': [vkupno_pivot[f'{m}_разлика'].mean() for m in ['Opus', 'Sonnet', 'Haiku']],
    'Медијана': [vkupno_pivot[f'{m}_разлика'].median() for m in ['Opus', 'Sonnet', 'Haiku']],
    'Ст. девијација': [vkupno_pivot[f'{m}_разлика'].std() for m in ['Opus', 'Sonnet', 'Haiku']],
    'MAE': [vkupno_pivot[f'{m}_разлика'].abs().mean() for m in ['Opus', 'Sonnet', 'Haiku']]
}).round(2)

diff_stats.set_index('AI Модел')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models = ['Opus', 'Sonnet', 'Haiku']

for i, model in enumerate(models):
    data = vkupno_pivot[f'{model}_разлика'].dropna()
    color = COLORS[model]
    
    axes[i].hist(data, bins=15, color=color, alpha=0.7, edgecolor='black')
    axes[i].axvline(x=0, color='black', linestyle='--', linewidth=2, label='Нула')
    axes[i].axvline(x=data.mean(), color='red', linestyle='-', linewidth=2, 
                   label=f'Просек: {data.mean():.1f}')
    
    axes[i].set_title(f'{model} - Професор', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Разлика (поени)')
    axes[i].set_ylabel('Број на проекти')
    axes[i].legend(fontsize=9)

plt.tight_layout()
plt.savefig('разлики_хистограм.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 7. Согласување по категории

Проектите се категоризирани според степенот на согласување помеѓу AI и Професор:
- **Блиско согласување (±10 поени)**: AI и Професор дале слична оценка
- **AI повисоко**: AI дал значајно повисока оценка (>10 поени разлика)
- **AI пониско**: AI дал значајно пониска оценка (>10 поени разлика)

In [ ]:
def categorize(diff, threshold=10):
    if abs(diff) <= threshold:
        return 'Блиско (±10)'
    elif diff > threshold:
        return 'AI повисоко'
    else:
        return 'AI пониско'

# Категоризирај
agreement_data = []
for model in ['Opus', 'Sonnet', 'Haiku']:
    categories = vkupno_pivot[f'{model}_разлика'].dropna().apply(categorize)
    counts = categories.value_counts()
    total = len(categories)
    agreement_data.append({
        'Модел': model,
        'Блиско (±10)': f"{counts.get('Блиско (±10)', 0)} ({counts.get('Блиско (±10)', 0)/total*100:.1f}%)",
        'AI повисоко': f"{counts.get('AI повисоко', 0)} ({counts.get('AI повисоко', 0)/total*100:.1f}%)",
        'AI пониско': f"{counts.get('AI пониско', 0)} ({counts.get('AI пониско', 0)/total*100:.1f}%)"
    })

pd.DataFrame(agreement_data).set_index('Модел')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

agreement_pct = []
for model in ['Opus', 'Sonnet', 'Haiku']:
    categories = vkupno_pivot[f'{model}_разлика'].dropna().apply(categorize)
    counts = categories.value_counts(normalize=True) * 100
    agreement_pct.append({
        'Модел': model,
        'Блиско (±10)': counts.get('Блиско (±10)', 0),
        'AI повисоко': counts.get('AI повисоко', 0),
        'AI пониско': counts.get('AI пониско', 0)
    })

agree_df = pd.DataFrame(agreement_pct).set_index('Модел')
agree_df.plot(kind='barh', stacked=True, ax=ax, color=['#27ae60', '#e74c3c', '#3498db'])

ax.set_title('Категории на согласување (праг: ±10 поени)', fontsize=13, fontweight='bold')
ax.set_xlabel('Процент на проекти')
ax.set_ylabel('AI Модел')
ax.legend(title='Категорија', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xlim(0, 100)

plt.tight_layout()
plt.savefig('согласување_категории.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 8. Споредба по тип на проект

Проектите се поделени на три типа:
- **noAI**: Студентот не користел AI алатки
- **onlyAI**: Студентот користел само AI алатки
- **hybrid**: Комбинација на двата пристапа

Овој график ги покажува просечните оценки по тип на проект за секој евалуатор.

In [ ]:
# Статистика по тип
type_stats = df.groupby(['project_type', 'evaluator'])['vkupno'].agg(['mean', 'count']).round(2)
type_stats = type_stats.unstack(level=0)
print("Просечна оценка по тип на проект:")
type_stats

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

type_means = df.groupby(['project_type', 'evaluator'])['vkupno'].mean().unstack()
type_means = type_means[['Професор', 'Opus', 'Sonnet', 'Haiku']]

type_means.plot(kind='bar', ax=ax, width=0.8, 
                color=[COLORS['Професор'], COLORS['Opus'], COLORS['Sonnet'], COLORS['Haiku']])

ax.set_title('Просечни оценки по тип на проект', fontsize=14, fontweight='bold')
ax.set_xlabel('Тип на проект')
ax.set_ylabel('Просечна оценка')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Евалуатор', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_ylim(0, 80)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', fontsize=9, padding=2)

plt.tight_layout()
plt.savefig('оценки_по_тип.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 9. Детален преглед на сите оценки

Оваа топлинска мапа ги прикажува сите оценки за секој проект од секој евалуатор.
- **Зелено**: Висока оценка
- **Жолто**: Средна оценка
- **Црвено**: Ниска оценка

In [ ]:
fig, ax = plt.subplots(figsize=(12, 14))

heatmap_data = vkupno_pivot.set_index('project_id')[['Професор', 'Opus', 'Sonnet', 'Haiku']]
heatmap_data = heatmap_data.dropna()

sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='RdYlGn', 
            center=50, vmin=0, vmax=100, ax=ax, cbar_kws={'label': 'Оценка'})

ax.set_title('Сите оценки по проект', fontsize=14, fontweight='bold')
ax.set_xlabel('Евалуатор')
ax.set_ylabel('ID на проект')

plt.tight_layout()
plt.savefig('топлинска_мапа.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 10. Сумарна табела

Оваа табела ги сумира клучните метрики за споредба на секој AI модел со Професорот.

In [ ]:
# Финална сумарна табела
summary_table = pd.DataFrame({
    'Метрика': [
        'Просечна оценка',
        'Корелација (Pearson r)',
        'MAE (просечна апс. грешка)',
        'Просечна разлика (AI-Проф)',
        'Согласување ±10 поени',
        'Согласување ±5 поени'
    ]
})

# Професор
summary_table['Професор'] = [
    f"{df[df['evaluator']=='Професор']['vkupno'].mean():.1f}",
    '—',
    '—',
    '—',
    '—',
    '—'
]

for model in ['Opus', 'Sonnet', 'Haiku']:
    valid = vkupno_pivot[['Професор', model]].dropna()
    diff = vkupno_pivot[f'{model}_разлика'].dropna()
    
    r = stats.pearsonr(valid['Професор'], valid[model])[0]
    mae = diff.abs().mean()
    avg_diff = diff.mean()
    agree_10 = (diff.abs() <= 10).mean() * 100
    agree_5 = (diff.abs() <= 5).mean() * 100
    
    summary_table[model] = [
        f"{df[df['evaluator']==model]['vkupno'].mean():.1f}",
        f"{r:.3f}",
        f"{mae:.2f}",
        f"{avg_diff:+.1f}",
        f"{agree_10:.1f}%",
        f"{agree_5:.1f}%"
    ]

summary_table.set_index('Метрика')

---

## Забелешки

- Колоната `tocnost` не е анализирана бидејќи за AI моделите е константна вредност (1.0)
- Формула за финална оценка: `vkupno = kviz × tocnost × metap`
- Сите корелации се статистички значајни (p < 0.001)